This notebook will be used to compare interpolation methods using synthetic SBES data with its original MBES point cloud as a validation scheme for comparing TIN, Natural Neighbor, IDW, RBF, and SVGPs for bathymetric reprocessing

In [ ]:
# TODO: GP class
# TODO: mbes raster sampling function to feed into the LINE diagnostics plots

# TODO: run for all 30 and summarize results

# TODO: Refine the grid generation for the IDW, RBF, and GP. Want to make sure it is confined to the same extent as the TIN and NATN interpolation without having to use those for clipping

In [ ]:
%reload_ext autoreload
%autoreload 2
import os
import sys
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt

from interpolators.TIN import TIN
from interpolators.NaturalNeighbor import NaturalNeighbor
from interpolators.KNeighbors_IDW import IDW
from interpolators.RBF import RBF

import processing_help
import numpy as np
from sklearn.cluster import MiniBatchKMeans

import warnings
# remove future warning of spline mindist since we aren't even providing it anyways
warnings.filterwarnings('ignore', message='.*mindist parameter.*', category=FutureWarning)     

import torch
import gpytorch
from torch.utils.data import TensorDataset, DataLoader

from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution
from gpytorch.variational import VariationalStrategy
from gpytorch.constraints import Interval, GreaterThan, LessThan
from gpytorch.priors import GammaPrior, LogNormalPrior, UniformPrior, NormalPrior
from gpytorch.means import Mean

from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator

import tqdm
import verde as vd
import rasterio
from rasterio.warp import transform_bounds
from rasterio.transform import from_origin
import contextily as cx

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
np.random.seed(1337)

In [ ]:
WORK_DIR = Path('/mnt/teamgroup/AutoDBM/Public')
MBES = WORK_DIR / 'Multibeam_Validation_Points_For_Distribution'
SBES = WORK_DIR / 'Synthetic_Singlebeam_Surveys_For_Distribution'
CRS = pd.read_csv(WORK_DIR / 'survey_CRS_list.csv')

mbes = list((WORK_DIR / 'Multibeam_Validation_Points_For_Distribution').glob('*.csv'))
sbes = list((WORK_DIR / 'Synthetic_Singlebeam_Surveys_For_Distribution').glob('*.csv'))
crs_data = WORK_DIR / 'survey_CRS_list.csv'

In [ ]:
# TODO: Go through and build a dataset off these 30 which include congressionally authorized depths
# this would allow us to better estimate dredge cuts and copmare across the interpolation methods

# NOTE: authorized depths probably vary between the individual reaches within each area


# Build a boundary catalog from SurveyJob layers, then match each survey to the best boundary by SurveyId
data_pairs = {}

def _norm(s):
    return "".join(ch for ch in str(s).upper() if ch.isalnum())

def _tokens(s):
    return {t for t in str(s).upper().replace("-", "_").split("_") if t}

# 1) Read all boundary candidates once
boundary_catalog = []
for gdb_file in WORK_DIR.glob("*/*.gdb"):
    try:
        sj = gpd.read_file(gdb_file, layer="SurveyJob")
    except Exception:
        continue

    if "SurveyId" not in sj.columns or sj.empty:
        continue

    # keep one record per SurveyId (first occurrence)
    for sid, grp in sj.groupby("SurveyId", dropna=True):
        boundary_catalog.append({
            "survey_id": str(sid),
            "survey_norm": _norm(sid),
            "survey_tokens": _tokens(sid),
            "gdb_file": gdb_file,
            "boundary": grp.copy()
        })

# 2) Match each survey from CRS table to best boundary
for _, items in pd.read_csv(crs_data).iterrows():
    survey = items["Survey"]

    sbes_match = [p for p in sbes if survey in str(p)]
    mbes_match = [p for p in mbes if survey in str(p)]
    if not sbes_match or not mbes_match:
        print(f"Missing SBES/MBES file for: {survey}")
        continue

    s_norm = _norm(survey)
    s_tokens = _tokens(survey)

    # exact normalized SurveyId match first
    exact = [c for c in boundary_catalog if c["survey_norm"] == s_norm]

    if exact:
        best = exact[0]
    else:
        # fallback: token overlap score
        scored = []
        for c in boundary_catalog:
            inter = len(s_tokens & c["survey_tokens"])
            denom = max(1, len(s_tokens))
            score = inter / denom
            scored.append((score, c))
        scored.sort(key=lambda x: x[0], reverse=True)
        best = scored[0][1] if scored and scored[0][0] > 0 else None

    if best is None:
        print(f"No boundary found for: {survey}")
        continue

    boundary = best["boundary"]
    data_pairs[survey] = [
        sbes_match[0],
        mbes_match[0],
        boundary,
    ]

    print(f"{survey} -> {best['survey_id']} ({best['gdb_file'].name})")

# Execute interpolation
1. TIN
2. Natural Neighbor
3. IDW
4. SVGP

RBF is available, but has finetuning implemented which may exceed memory for large MBES point clouds


In [ ]:
# TODO: implement block CV using Verde blocking. Probably only 5-fold to save on compute time. If fast enough, we may try 10-fold
# NOTE: Need to make sure that the deterministic methods are still predicting within the entire boundary

In [ ]:
# It seemse like the mbes and sbes dataframes do not overlap at all, so we can just directly sample the mbes from the resulting bathy surfaces.

# TODO: Play with the removing points within the tracklines and removing tracklines entirely; the sampling strategy plays a larger role in the accuracy of the bathymetric surface than the interpolation
# though, the interpolation plays a large role in how accurate the surface is to reality (e.g., interpolated depths between soundings and artifacts impacting volume estimation)

survey = list(data_pairs.keys())[6]

test_pair = data_pairs[survey]
sbes_data = pd.read_csv(test_pair[0])
mbes_data = pd.read_csv(test_pair[1])
boundary = test_pair[2]
data_crs = boundary.crs

sbes_gdf = gpd.GeoDataFrame(
        sbes_data.copy(),
        geometry=gpd.points_from_xy(sbes_data["X_ft"], sbes_data["Y_ft"]),
        crs=data_crs
    )

mbes_gdf = gpd.GeoDataFrame(
        mbes_data.copy(),
        geometry=gpd.points_from_xy(mbes_data["X_ft"], mbes_data["Y_ft"]),
        crs=data_crs
    )

OUT_DIR = WORK_DIR / survey
os.makedirs(OUT_DIR, exist_ok=True)

sbes_x = sbes_gdf.X_ft
sbes_y = sbes_gdf.Y_ft
sbes_z = sbes_gdf.Elevation_ft

mbes_x = mbes_gdf.X_ft
mbes_y = mbes_gdf.Y_ft
mbes_z = mbes_gdf.Elevation_ft

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10, 5))

# Histogram for sbes_z
ax[0].hist(sbes_z, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[0].set_xlabel('Elevation (ft)', fontsize=12)
ax[0].set_ylabel('Frequency', fontsize=12)
ax[0].set_title(f'SBES Elevation Distribution (n={len(sbes_z):,})', fontsize=14)
ax[0].grid(axis='y', alpha=0.3)

# Histogram for mbes_z
ax[1].hist(mbes_z, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[1].set_xlabel('Elevation (ft)', fontsize=12)
ax[1].set_ylabel('Frequency', fontsize=12)
ax[1].set_title(f'MBES Elevation Distribution (n={len(mbes_z):,})', fontsize=14)
ax[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSBES Elevation Statistics (n={len(sbes_z):,}):")
print(f"  Mean: {np.mean(sbes_z):.2f} ft")
print(f"  Std: {np.std(sbes_z):.2f} ft")
print(f"  Min: {np.min(sbes_z):.2f} ft")
print(f"  Max: {np.max(sbes_z):.2f} ft")

print(f"\nMBES Elevation Statistics (n={len(mbes_z):,}):")
print(f"  Mean: {np.mean(mbes_z):.2f} ft")
print(f"  Std: {np.std(mbes_z):.2f} ft")
print(f"  Min: {np.min(mbes_z):.2f} ft")
print(f"  Max: {np.max(mbes_z):.2f} ft")

# **Generate bathymetric surfaces with various interpolation methods**
1. TIN (as baseline)
2. Natural Neighbor (using dual graph of Delaunay Triangulation, Voronoi/Theissian Polygons)
3. Inverse Distance Weighting
4. Radial Basis Functions
5. Gaussian Process Regression (Stochastic Variational)

**Error metrics will be calculated by sampling the left out MBES points directly from the resulting bathymetric surfaces.** Block cross-validation is a more robust method for calculating error metrics, but for now (10 JUN 2026) sampling left out points will be sufficient.

### *Deterministics methods first*

In [ ]:
# horizontal resolution of the resulting bathymetric surface. This is in US survey ft
# 10 ft resolution matches the inputs used in tools such as the Corps Shoaling Analysis Tool (CSAT)
grid_res = 10.0 # ft

plots = True               # if the outputs of the interpolation methods will be showcased in this notebook using Matplotlib

TINinterpolator = TIN(
        surveyId = survey,
        survey_path = OUT_DIR,
        x=sbes_x, y=sbes_y, z=sbes_z,
        survey_boundary = boundary,
        echosounder = "SBES",
        grid_res=grid_res,
        plot_outputs=plots,
        n_jobs=8
    )

tin_path = TINinterpolator.generate()
_,_ = processing_help.validate_bathymetric_surface(tin_path, mbes_x, mbes_y, mbes_z)

In [ ]:
# Generate the same tif but with all MBES points as a "ground truth" raster

TINinterpolator = TIN(
        surveyId = survey,
        survey_path = OUT_DIR,
        x=mbes_x, y=mbes_y, z=mbes_z,
        survey_boundary = boundary,
        echosounder = "MBES",
        grid_res=grid_res,
        plot_outputs=False,
        n_jobs=8
    )

tin_path = TINinterpolator.generate()

In [ ]:
NatNinterpolator = NaturalNeighbor(
        surveyId = survey,
        survey_path = OUT_DIR,
        x=sbes_x, y=sbes_y, z=sbes_z, 
        survey_boundary = boundary,
        grid_res=grid_res,
        plot_outputs=plots,
        n_jobs=8
    )
    
natn_path = NatNinterpolator.generate()
_,_ = processing_help.validate_bathymetric_surface(natn_path, mbes_x, mbes_y, mbes_z)

In [ ]:
IDWinterpolator = IDW(
        surveyId = survey,
        survey_path = OUT_DIR,
        x=sbes_x, y=sbes_y, z=sbes_z,
        survey_boundary = boundary,
        grid_res=grid_res,
        plot_outputs=plots,
        n_jobs=8
    )
    
idw_path = IDWinterpolator.generate()
_,_ = processing_help.validate_bathymetric_surface(idw_path, mbes_x, mbes_y, mbes_z)

In [ ]:
RBFinterpolator = RBF(
        surveyId = survey,
        survey_path = OUT_DIR,
        x=sbes_x, y=sbes_y, z=sbes_z, 
        survey_boundary = boundary,
        grid_res=grid_res,
        plot_outputs=plots,
        n_jobs=8
    )

rbf_path = RBFinterpolator.generate(auto_tune=True)
_,_ = processing_help.validate_bathymetric_surface(rbf_path, mbes_x, mbes_y, mbes_z)

### *Geostatistical / GP methods second*

- Some additional pre-processing steps may be needed for GP regression. GPs assume that data have a mean centered at zero 

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(20, 5))

sbes_z_norm = (sbes_z - np.mean(sbes_z)) / np.std(sbes_z)
mbes_z_norm = (mbes_z - np.mean(mbes_z)) / np.std(mbes_z)

# Histogram for sbes_z_norm
ax[0].hist(sbes_z_norm, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[0].set_xlabel('Standardized Elevation', fontsize=12)
ax[0].set_ylabel('Frequency', fontsize=12)
ax[0].set_title(f'SBES Standardized Elevation Distribution (n={len(sbes_z_norm):,})', fontsize=14)
ax[0].grid(axis='y', alpha=0.3)

# Histogram for mbes_z_norm
ax[1].hist(mbes_z_norm, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[1].set_xlabel('Standardized Elevation', fontsize=12)
ax[1].set_ylabel('Frequency', fontsize=12)
ax[1].set_title(f'MBES Standardized Elevation Distribution (n={len(mbes_z_norm):,})', fontsize=14)
ax[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


# Print summary statistics
print(f"\nSBES Elevation Statistics (n={len(sbes_z_norm):,}):")
print(f"  Mean: {np.mean(sbes_z_norm):.2f} ft")
print(f"  Std: {np.std(sbes_z_norm):.2f} ft")
print(f"  Min: {np.min(sbes_z_norm):.2f} ft")
print(f"  Max: {np.max(sbes_z_norm):.2f} ft")

print(f"\nMBES Elevation Statistics (n={len(mbes_z_norm):,}):")
print(f"  Mean: {np.mean(mbes_z_norm):.2f} ft")
print(f"  Std: {np.std(mbes_z_norm):.2f} ft")
print(f"  Min: {np.min(mbes_z_norm):.2f} ft")
print(f"  Max: {np.max(mbes_z_norm):.2f} ft")

In [ ]:
# We can do some fun detrending to remove portions of the data which may be contributing to non-stationarity and non-linearity within the soundings

sbes_xy_raw = np.column_stack([sbes_x, sbes_y])
mbes_xy_raw = np.column_stack([mbes_x, mbes_y])

sbes_chain = vd.Chain([
    ("reduce", vd.BlockMean(spacing=50)),   # adjust spacing to your CRS units
    ("spline", vd.Spline(damping=1e-4)),
]).fit((sbes_xy_raw[:, 0], sbes_xy_raw[:, 1]), sbes_z)

# chain.fit((sbes_xy_raw[:, 0], sbes_xy_raw[:, 1]), sbes_z)
sbes_trend_pred = sbes_chain.predict((sbes_xy_raw[:, 0], sbes_xy_raw[:, 1]))   # evaluate at full resolution
sbes_residuals = sbes_z - sbes_trend_pred


mbes_chain = vd.Chain([
    ("reduce", vd.BlockMean(spacing=50)),   # adjust spacing to your CRS units
    ("spline", vd.Spline(damping=1e-4)),
]).fit((mbes_xy_raw[:, 0], mbes_xy_raw[:, 1]), mbes_z)

# chain.fit((sbes_xy_raw[:, 0], sbes_xy_raw[:, 1]), sbes_z)
mbes_trend_pred = mbes_chain.predict((mbes_xy_raw[:, 0], mbes_xy_raw[:, 1]))   # evaluate at full resolution
mbes_residuals = mbes_z - mbes_trend_pred

fig, ax = plt.subplots(1, 2, figsize=(20, 5))

# Histogram for sbes_residuals
ax[0].hist(sbes_residuals, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[0].set_xlabel('Detrended Elevation', fontsize=12)
ax[0].set_ylabel('Frequency', fontsize=12)
ax[0].set_title(f'SBES Trend Elevation Distribution (n={len(sbes_residuals):,})', fontsize=14)
ax[0].grid(axis='y', alpha=0.3)

ax[1].hist(mbes_residuals, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[1].set_xlabel('Detrended Elevation', fontsize=12)
ax[1].set_ylabel('Frequency', fontsize=12)
ax[1].set_title(f'MBES Trend Elevation Distribution (n={len(mbes_residuals):,})', fontsize=14)
ax[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSBES Detrended Elevation Statistics (n={len(sbes_residuals):,}):")
print(f"  Mean: {np.mean(sbes_residuals):.2f} ft")
print(f"  Std: {np.std(sbes_residuals):.2f} ft")
print(f"  Min: {np.min(sbes_residuals):.2f} ft")
print(f"  Max: {np.max(sbes_residuals):.2f} ft")

print(f"\nMBES Detrended Elevation Statistics (n={len(mbes_residuals):,}):")
print(f"  Mean: {np.mean(mbes_residuals):.2f} ft")
print(f"  Std: {np.std(mbes_residuals):.2f} ft")
print(f"  Min: {np.min(mbes_residuals):.2f} ft")
print(f"  Max: {np.max(mbes_residuals):.2f} ft")

In [ ]:
# now we can do more fun stuff with the detrended elevation data and standardize it for better GP regression

sbes_residuals_norm = (sbes_residuals - np.mean(sbes_residuals)) / np.std(sbes_residuals)
mbes_residuals_norm = (mbes_residuals - np.mean(mbes_residuals)) / np.std(mbes_residuals)


fig, ax = plt.subplots(1, 2, figsize=(20, 5))

# Histogram for sbes_residuals_norm
ax[0].hist(sbes_residuals_norm, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[0].set_xlabel('Standardized Detrended Elevation', fontsize=12)
ax[0].set_ylabel('Frequency', fontsize=12)
ax[0].set_title(f'SBES Standardized Trend Elevation Distribution (n={len(sbes_residuals_norm):,})', fontsize=14)
ax[0].grid(axis='y', alpha=0.3)

ax[1].hist(mbes_residuals_norm, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[1].set_xlabel('Detrended Elevation', fontsize=12)
ax[1].set_ylabel('Frequency', fontsize=12)
ax[1].set_title(f'MBES Standardized Trend Elevation Distribution (n={len(mbes_residuals_norm):,})', fontsize=14)
ax[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSBES Detrended Elevation Statistics (n={len(sbes_residuals_norm):,}):")
print(f"  Mean: {np.mean(sbes_residuals_norm):.2f} ft")
print(f"  Std: {np.std(sbes_residuals_norm):.2f} ft")
print(f"  Min: {np.min(sbes_residuals_norm):.2f} ft")
print(f"  Max: {np.max(sbes_residuals_norm):.2f} ft")

print(f"\nSBES Detrended Elevation Statistics (n={len(mbes_residuals_norm):,}):")
print(f"  Mean: {np.mean(mbes_residuals_norm):.2f} ft")
print(f"  Std: {np.std(mbes_residuals_norm):.2f} ft")
print(f"  Min: {np.min(mbes_residuals_norm):.2f} ft")
print(f"  Max: {np.max(mbes_residuals_norm):.2f} ft")

In [ ]:
# should be 10 ft spatial resolution for the outputs
grid_res = 10.0 

# train on all sbes points
X_all = torch.as_tensor(sbes_xy_raw, dtype=torch.float32)
y_all = torch.as_tensor(sbes_residuals_norm, dtype=torch.float32)

# Standardize the raw coordinates as they are in a different scale than the elevation
x_mean = X_all.mean(0)
x_std = X_all.std(0).clamp_min(1e-6)
X_all_scaled = (X_all - x_mean) / x_std

if torch.cuda.is_available():
    X_all_scaled = X_all_scaled.cuda().float()
    y_all = y_all.cuda().float()
sbes_xy_raw = np.asarray(sbes_xy_raw, dtype=np.float32)
x_min, x_max = float(sbes_xy_raw[:, 0].min()), float(sbes_xy_raw[:, 0].max())
y_min, y_max = float(sbes_xy_raw[:, 1].min()), float(sbes_xy_raw[:, 1].max())

nx = int(np.ceil((x_max - x_min) / grid_res)) + 1
ny = int(np.ceil((y_max - y_min) / grid_res)) + 1
x_grid = np.linspace(x_min, x_min + (nx - 1) * grid_res, nx, dtype=np.float32)
y_grid = np.linspace(y_min, y_min + (ny - 1) * grid_res, ny, dtype=np.float32)

transform = from_origin(
    x_min - grid_res / 2,
    y_max + grid_res / 2,
    grid_res, grid_res
)

grid_xx, grid_yy = np.meshgrid(x_grid, y_grid)  # shape (ny, nx) by default
grid_points = torch.stack([
    torch.tensor(grid_xx.flatten(), dtype=torch.float32),
    torch.tensor(grid_yy.flatten(), dtype=torch.float32)
], dim=1)

grid_points_s = (grid_points - x_mean.cpu()) / x_std.cpu()

In [ ]:
all_train_dataset = TensorDataset(X_all_scaled, y_all)
# all_train_loader = DataLoader(all_train_dataset, batch_size=1024, shuffle=True)\

all_train_loader = DataLoader(
    all_train_dataset,
    batch_size=2048, 
    shuffle=True,
    num_workers=0 # Keep at 0 since data is already on GPU
)

In [ ]:
class TINMean(Mean):
    def __init__(self, xy, z):
        super().__init__()
        xy = xy.detach().cpu().numpy()
        z = z.detach().cpu().numpy()
        self._lin = LinearNDInterpolator(xy, z)
        self._near = NearestNDInterpolator(xy, z)

    def forward(self, x):
        shape = x.shape[:-1]
        xy = x.detach().cpu().numpy().reshape(-1, x.shape[-1])
        m = self._lin(xy)
        nan = np.isnan(m)
        if nan.any():
            m[nan] = self._near(xy[nan])
        return torch.as_tensor(m, dtype=x.dtype, device=x.device).reshape(shape)

class GPModel(ApproximateGP):
    def __init__(self, inducing_points, train_x, train_z):
        variational_distribution = CholeskyVariationalDistribution(inducing_points.size(0))
        variational_strategy = VariationalStrategy(
            self, inducing_points, variational_distribution,
            learn_inducing_locations=True,
        )
        super().__init__(variational_strategy)

        # TODO: Try out different mean functions
        # Maybe even try some of the means with first and second derivatives?
        
        # self.mean_module = TINMean(train_x, train_z)      # NOTE: if using the TIN mean, the GP is smoothing the TIN instead of learning to predict the depth. So GP is used to remove artifacats and noise generated by TIN interpolation
        self.mean_module = gpytorch.means.ZeroMean()
        # self.mean_module = gpytorch.means.ConstantMean()      # NOTE: this one is probably redundant since pre-processing has been done to confirm a mean of zero
        # self.mean_module = gpytorch.means.LinearMean(input_size=2)

        ext = (train_x.max(dim=0).values - train_x.min(dim=0).values).detach().cpu()
        dmin, dmax = float(ext.min()), float(ext.max())

        self.covar_short = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.MaternKernel(
                nu=1.5, ard_num_dims=2,
                lengthscale_constraint=Interval(dmin * 0.005, dmin * 0.10),
            ),
            outputscale_prior=GammaPrior(2.0, 4.0),
        )
        self.covar_long = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.MaternKernel(
                nu=2.5, ard_num_dims=2,
                lengthscale_constraint=Interval(dmin * 0.10, dmax * 0.50),  
            ),
            outputscale_prior=GammaPrior(2.0, 4.0),
        )

        self.covar_short.base_kernel.lengthscale = dmin * 0.03
        self.covar_long.base_kernel.lengthscale = dmin * 0.20
        self.covar_short.outputscale = 0.1
        self.covar_long.outputscale = 0.1

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_short(x) + self.covar_long(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

In [ ]:
# TODO: Make the number of inducing points an input when pushing to HPC

X_unique = torch.unique(X_all_scaled, dim=0)

# TODO: refine this so that if there are less than 3000 points, an exact GP is used instead of an SVGP
# TODO: also, maybe have it increase for huge datasets. 

def init_inducing_points(X, n_inducing=3000):
    # Use MiniBatchKMeans on CPU for speed, then move back to the active device
    kmeans = MiniBatchKMeans(n_clusters=n_inducing, batch_size=10000, n_init=3, random_state=42)
    kmeans.fit(X.detach().cpu().numpy())
    centers = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32)

    # Tiny perturbation helps avoid near-duplicate inducing locations that can destabilize covariance matrix
    centers = centers + 1e-5 * torch.randn_like(centers)
    return centers.to(X.device)

num_inducing = min(5000, X_unique.size(0))
inducing_points = init_inducing_points(X_unique, num_inducing)

In [ ]:
# train_x_all, train_y_all = all_train_loader.dataset.tensors   # (N,2) coords, (N,) depths

# model = GPModel(inducing_points=inducing_points,
#                 train_x=train_x_all, train_z=train_y_all)

train_x_all, train_y_all = all_train_loader.dataset.tensors

model = GPModel(inducing_points=inducing_points, train_x=train_x_all, train_z=train_y_all)

likelihood = gpytorch.likelihoods.GaussianLikelihood(
    noise_constraint=gpytorch.constraints.GreaterThan(1e-3),
    noise_prior=GammaPrior(2.0, 50.0),
)

if torch.cuda.is_available():
    model = model.cuda()
    likelihood = likelihood.cuda()

# Explicit, non-fragile parameter split.
variational_optimizer = gpytorch.optim.NGD(
    model.variational_parameters(),
    num_data=train_y_all.size(0),
    lr=0.01,
)
hyperparameter_optimizer = torch.optim.Adam(
    [{"params": model.hyperparameters()}, {"params": likelihood.parameters()}],
    lr=0.01,
)

mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=train_y_all.size(0))

In [ ]:
num_epochs = 50

model.train()
likelihood.train()

optimizer = torch.optim.Adam([
    {'params': model.parameters()},
    {'params': likelihood.parameters()},
], lr=0.01)

# Our loss object. We're using the VariationalELBO for variational inference
mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=y_all.size(0))

epochs_iter = tqdm.notebook.tqdm(range(num_epochs), desc="Epoch")
with gpytorch.settings.cholesky_jitter(1e-4), gpytorch.settings.variational_cholesky_jitter(1e-4), gpytorch.settings.cholesky_max_tries(10):
    for i in epochs_iter:
        # Within each iteration, we will go over each minibatch of data
        minibatch_iter = tqdm.notebook.tqdm(all_train_loader, desc="Minibatch", leave=False)
        for x_batch, y_batch in minibatch_iter:
            optimizer.zero_grad()
            output = model(x_batch)
            loss = -mll(output, y_batch)
            minibatch_iter.set_postfix(loss=loss.item())
            loss.backward()
            optimizer.step()

In [ ]:
from shapely import contains_xy

model.eval()
likelihood.eval()
pred_means = []
pred_stds = []

with torch.no_grad(), gpytorch.settings.fast_pred_var():
    for i in range(0, grid_points_s.size(0), 2048):
        batch = grid_points_s[i:i+2048].cuda() if torch.cuda.is_available() else grid_points_s[i:i+2048]
        # preds = likelihood(model(batch))
        preds = model(batch)
        pred_means.append(preds.mean.cpu())
        pred_stds.append(preds.stddev.cpu())

pred_means = torch.cat(pred_means)
pred_stds = torch.cat(pred_stds)

# Convert back to real units
z_grid = pred_means * np.std(sbes_residuals) + np.mean(sbes_residuals)
z_uncertainty = pred_stds * np.std(sbes_residuals)

trend_pred_grid = sbes_chain.predict((grid_points[:, 0].numpy(), grid_points[:, 1].numpy()))
z_grid_true = z_grid.numpy() + trend_pred_grid

# Reshape to 2D grid'
z_surface = z_grid_true.reshape(grid_xx.shape)
z_unc_surface = z_uncertainty.reshape(grid_xx.shape)

grid_np = grid_points.numpy()
mask = contains_xy((boundary).union_all(), grid_np[:, 0], grid_np[:, 1])

# Apply mask to predictions.
z_grid_masked = z_grid_true
z_unc_masked = z_uncertainty
z_grid_masked[~mask] = np.nan
z_unc_masked[~mask] = np.nan

# Reshape back to 2D arrays aligned with transform/from_origin.
z_surface_masked = z_grid_masked.reshape(grid_xx.shape)
z_unc_surface_masked = z_unc_masked.reshape(grid_xx.shape)
 
z_raster = np.flipud(z_surface_masked).astype(np.float32)
z_unc_raster = np.flipud(z_unc_surface_masked).astype(np.float32)

In [ ]:
geojson = Path(os.getcwd()).parent / 'National_Channel_Framework_-6742069550782745925.geojson'
ncf = gpd.read_file(geojson)

ncf_polygons_3857 = ncf.to_crs(epsg=3857)

# Reproject raster extent from survey CRS to Web Mercator.
x_min_3857, y_min_3857, x_max_3857, y_max_3857 = transform_bounds(
    data_crs, "EPSG:3857", x_min, y_min, x_max, y_max
)
extent_3857 = [x_min_3857, x_max_3857, y_min_3857, y_max_3857]

# plot predictions and prediction uncertainty
fig, axes = plt.subplots(1, 2, figsize=(18, 9), dpi=150)

im1 = axes[0].imshow(
    np.flipud(z_raster), origin="lower", extent=extent_3857,
    cmap="terrain", alpha=0.7, zorder=2
)

ncf_polygons_3857.plot(ax=axes[0], facecolor='none', edgecolor='red',
                       linewidth=2, zorder=3)
cx.add_basemap(axes[0], source=cx.providers.OpenStreetMap.Mapnik,
               crs="EPSG:3857", zorder=1)
axes[0].set_xlim(extent_3857[0], extent_3857[1])
axes[0].set_ylim(extent_3857[2], extent_3857[3])
axes[0].set_title("SVGP Interpolation (Depth)")
axes[0].set_xlabel("Easting (m, Web Mercator)")
axes[0].set_ylabel("Northing (m, Web Mercator)")
fig.colorbar(im1, ax=axes[0], label="Depth (ft)", shrink=0.8)


im2 = axes[1].imshow(
    np.flipud(z_unc_raster), origin="lower", extent=extent_3857,
    cmap="magma_r", alpha=0.7, zorder=2
)

ncf_polygons_3857.plot(ax=axes[1], facecolor='none', edgecolor='red',
                       linewidth=2, zorder=3)
cx.add_basemap(axes[1], source=cx.providers.OpenStreetMap.Mapnik,
               crs="EPSG:3857", zorder=1)
axes[1].set_xlim(extent_3857[0], extent_3857[1])
axes[1].set_ylim(extent_3857[2], extent_3857[3])
axes[1].set_title("SVGP Predictive Uncertainty (1σ)")
axes[1].set_xlabel("Easting (m, Web Mercator)")
axes[1].set_ylabel("Northing (m, Web Mercator)")
fig.colorbar(im2, ax=axes[1], label="Uncertainty (ft)", shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Extract raster predictions at the original point locations
# Uses nearest raster cell lookup based on the existing affine transform.
xs = mbes_xy_raw[:, 0]
ys = mbes_xy_raw[:, 1]

rows, cols = rasterio.transform.rowcol(transform, xs, ys)

z_interp_points = np.full(xs.shape[0], np.nan, dtype=np.float32)

rows = np.asarray(rows)
cols = np.asarray(cols)

in_bounds = (
    (rows >= 0) & (rows < z_raster.shape[0]) &
    (cols >= 0) & (cols < z_raster.shape[1])
)

z_interp_points[in_bounds] = z_raster[rows[in_bounds], cols[in_bounds]]

In [ ]:
from scipy.spatial import ConvexHull
import alphashape

sbes_hull = ConvexHull(sbes_xy_raw)
mbes_hull = ConvexHull(mbes_xy_raw)

fig, ax = plt.subplots(1, 2, figsize=(10, 10))

ax[0].scatter(sbes_xy_raw[:, 0], sbes_xy_raw[:, 1], s=1, alpha=0.3, label="SBES points")

for simplex in sbes_hull.simplices:
    ax[0].plot(
        sbes_xy_raw[simplex, 0],
        sbes_xy_raw[simplex, 1],
        "r-",
        linewidth=2,
    )

ax[0].set_aspect("equal", adjustable="box")
ax[0].set_title("Convex Hull of SBES Points")
ax[0].set_xlabel("X")
ax[0].set_ylabel("Y")
ax[0].legend()

ax[1].scatter(mbes_xy_raw[:, 0], mbes_xy_raw[:, 1], s=1, alpha=0.3, label="MBES points")

for simplex in mbes_hull.simplices:
    ax[1].plot(
        mbes_xy_raw[simplex, 0],
        mbes_xy_raw[simplex, 1],
        "r-",
        linewidth=2,
    )

ax[1].set_aspect("equal", adjustable="box")
ax[1].set_title("Convex Hull of MBES Points")
ax[1].set_xlabel("X")
ax[1].set_ylabel("Y")
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
sbes_alphashape = alphashape.alphashape(sbes_xy_raw, 0.025)
mbes_alphashape = alphashape.alphashape(mbes_xy_raw, 0.025)

fig, ax = plt.subplots(1, 2, figsize=(10, 10))

ax[0].scatter(sbes_xy_raw[:, 0], sbes_xy_raw[:, 1], s=1, alpha=0.3, label="SBES points")

if sbes_alphashape.geom_type == "Polygon":
    x, y = sbes_alphashape.exterior.xy
    ax[0].plot(x, y, "r-", linewidth=2, label="Alpha shape")
elif sbes_alphashape.geom_type == "MultiPolygon":
    for geom in sbes_alphashape.geoms:
        x, y = geom.exterior.xy
        ax[0].plot(x, y, "r-", linewidth=2)

ax[0].set_aspect("equal", adjustable="box")
ax[0].set_title("Alpha Shape of SBES Points")
ax[0].set_xlabel("X")
ax[0].set_ylabel("Y")
ax[0].legend()

ax[1].scatter(mbes_xy_raw[:, 0], mbes_xy_raw[:, 1], s=1, alpha=0.3, label="MBES points")

if mbes_alphashape.geom_type == "Polygon":
    x, y = mbes_alphashape.exterior.xy
    ax[1].plot(x, y, "r-", linewidth=2, label="Alpha shape")
elif mbes_alphashape.geom_type == "MultiPolygon":
    for geom in mbes_alphashape.geoms:
        x, y = geom.exterior.xy
        ax[1].plot(x, y, "r-", linewidth=2)

ax[1].set_aspect("equal", adjustable="box")
ax[1].set_title("Alpha Shape of MBES Points")
ax[1].set_xlabel("X")
ax[1].set_ylabel("Y")
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
for simplex in alphashape.alphasimplices(sbes_xy_raw):
    print(simplex)

In [ ]:
# TODO: do this for each of the generated rasters. Probably just adding this on to the validate_bathymetric_surface function would be easiest

# Convenience wrapper using your existing notebook variables
def run_current_prediction_diagnostics(**kwargs):
    return processing_help.run_line_diagnostics(
        z_pred=z_interp_points,       
        z_true=np.array(mbes_z),
        xy_coords=mbes_xy_raw,
        **kwargs,
    )

metrics = run_current_prediction_diagnostics(make_plots=True, moran_sample=None, plot_sample=None)

In [ ]:
# export bathymetric surface

meta = {
    'driver': 'GTiff',
    'height': z_raster.shape[0],
    'width': z_raster.shape[1],
    'count': 1,
    'dtype': 'float32',
    'crs': data_crs,
    'transform': transform,
    'nodata': -9999,
    'tiled': True,
    'blockxsize': 512,
    'blockysize': 512,
    'compress': 'deflate',
    'predictor': 3,
    'zlevel': 6
}

with rasterio.open(OUT_DIR / 'SVGP_bathy.tif', 'w', **meta) as dst:
    dst.write(z_raster, 1)
    dst.build_overviews([2, 4, 8, 16, 32], rasterio.enums.Resampling.average)

with rasterio.open(OUT_DIR / 'SVGP_uncertainty.tif', 'w', **meta) as dst:
    dst.write(z_unc_raster, 1)
    dst.build_overviews([2, 4, 8, 16, 32], rasterio.enums.Resampling.average)

# make sure to run the cross-validation scheme like the deterministic methods

df , stats = processing_help.validate_bathymetric_surface(OUT_DIR / 'SVGP_bathy.tif', mbes_x, mbes_y, mbes_z)

In [ ]:
df

In [ ]:
mbes_xy_raw

In [ ]:
# Correctly stack X and Y columns from df into (N,2) array
coords = df[['X', 'Y']].to_numpy()
# If you need float32 for torch: coords = df[['X','Y']].to_numpy(dtype=np.float32)

In [ ]:
processing_help.run_line_diagnostics(
        z_pred=np.array(df.Predicted_Depth),       
        z_true=np.array(df.True_Depth),
        xy_coords=mbes_xy_raw,
        make_plots=True, moran_sample=None, plot_sample=None
    )

metrics = run_current_prediction_diagnostics()

# Next, we can compute dredge volumes within the national channel framework reaches covered by each survey

In [ ]:
bathy_tifs = list(OUT_DIR.glob("*bathy.tif"))
tgt_ncf = ncf[ncf['channelareaidfk'].str.contains(f'{survey[:9]}')]
tgt_ncf

In [ ]:
bathy_tifs

In [ ]:
methods = {"MBES": bathy_tifs[1] ,'GP': bathy_tifs[-1], 'TIN': bathy_tifs[2], 'NATN': bathy_tifs[-2], 'IDW': bathy_tifs[0], 'RBF': bathy_tifs[3]}

# NOTE: sometimes the RBF tif is not generated due to computational constraints when the optimal number of neighboring points is parameterized as None
# in that case we must remove the RBF from the methods. Processing on Jean which has much more memory for CPU based processing may alleviate this
# methods = {"MBES": bathy_tifs[1] ,'GP': bathy_tifs[-1], 'TIN': bathy_tifs[2], 'NATN': bathy_tifs[-2], 'IDW': bathy_tifs[0]}

In [ ]:
# TODO: integrate the GP uncertainty surface within the volume estimate to have an uncertainty value for the dredge cuts

unit = "yd"

reach_vols = {}

for id, geom in tgt_ncf.iterrows():
    reachid = geom.channelreachidpk
    reach_auth_depth = geom.depthauthorized
    # print(reach_auth_depth)
    reach_vols[reachid] = {name: processing_help.compute_volume(path, tgt_ncf, auth_depth = reach_auth_depth, unit = unit) for name, path in methods.items()}

for reach, volumes in reach_vols.items():
    print(f'\n{reach}')
    print("_"*50)
    for key, value in volumes.items():
        if key == 'GP':
            print(f'{key}: {value[0]:,.1f}, {value[1]:,.1f}, {value[2]:,.1f} {unit}³')
        else:
            print(f'{key}: {value:,.1f} {unit}³')

In [ ]:
# Differences
for reach, volumes in reach_vols.items():
    print(f'\n{reach}')
    print(f"_"*50)
    if 'RBF' in methods.keys():
        for a, b in [('MBES', 'TIN'), ('MBES', 'NATN'), ('MBES', 'IDW'), ('MBES', 'RBF'), ('MBES', 'GP')]:
            if b == 'GP':
                diff_low = volumes[a] - volumes[b][0]
                diff = volumes[a] - volumes[b][1]
                diff_up = volumes[a] - volumes[b][2]
                print(f'{a} - {b}: {diff_low:,.1f}, {diff:,.1f}, {diff_up:,.1f} {unit}³')
            else:
                diff = volumes[a] - volumes[b]
                print(f'{a} - {b}: {diff:,.1f} {unit}³')
    else:
        for a, b in [('MBES', 'TIN'), ('MBES', 'NATN'), ('MBES', 'IDW'), ('MBES', 'GP')]:
            diff = volumes[a] - volumes[b]
            print(f'{a} - {b}: {diff:,.1f} {unit}³')

# Additionally to the bulk statistics and the cut estimate differences we can further decide on best interpolation methods using:
1. Akaike Information Criterion (AIC) comparison
2. Block cross-validation

In [ ]:
def _calculate_aic(self, model, bin_center, gamma):
        """
        Calculate AIC for a fitted variogram model.
        
        Parameters
        ----------
        model : gstools.CovModel
            Fitted variogram model
        bin_center : array
            Bin centers from empirical variogram
        gamma : array
            Empirical variogram values
        
        Returns
        -------
        float
            AIC score (lower is better)
        """
        # Get predicted values from the fitted model
        gamma_pred = model.variogram(bin_center)
        
        # Calculate residuals
        residuals = gamma - gamma_pred
        
        # Calculate RSS (Residual Sum of Squares)
        rss = np.sum(residuals**2)
        
        # Number of data points
        n = len(bin_center)
        
        # Number of parameters in the model
        # Most gstools models have 3 parameters: var (variance), len_scale (range), nugget
        # Some models may have additional parameters
        k = 3  # Default: variance, length scale, nugget
        
        # Handle models with additional parameters
        model_name = model.__class__.__name__.lower()
        if 'stable' in model_name:
            k = 4  # Includes shape parameter (alpha)
        elif 'matern' in model_name:
            k = 4  # Includes smoothness parameter (nu)
        
        # Calculate AIC using the formula: AIC = n * ln(RSS/n) + 2*k
        # Add small epsilon to avoid log(0)
        if rss < 1e-10:
            rss = 1e-10
        
        aic = n * np.log(rss / n) + 2 * k
        
        # Apply small sample correction (AICc) if n/k < 40
        if n / k < 40:
            aic = aic + (2 * k * (k + 1)) / (n - k - 1)
        
        return aic